# 1109 47일차

## 1. 지연 시간

호출 한 번에 걸리는 시간을 여러 번 재서 평균으로 보는 것 (`f12_latency`, `f15_prefill_decode`, `f18_compare`)

한 번만 재면 그때 서버 상태에 따라 값이 흔들림. 같은 호출을 반복해 평균을 내야 견줄 수 있는 수치가 됨

1. `times = []`과 `for i in range(5):` : 같은 호출을 5번 반복해서 시간을 모음
2. `elapsed = end - start` : 호출 전후 `perf_counter` 차이
3. `sum(times) / len(times)` : 모은 시간의 평균
4. `prompts` 리스트 : 길이가 다른 질문을 차례로 보내 시간을 견줌

- 입력 길이를 몇 배로 늘려도 전체 시간은 거의 그대로임
- **전체 시간은 입력 길이가 아니라 출력 토큰 수가 정함**
- 입력을 읽는 단계(prefill)는 한 번에 처리되고, 출력은 토큰을 하나씩 만드는(decode) 단계라 토큰 수만큼 반복함
- `max_tokens`에 걸려 출력이 같은 길이로 끝나면 질문이 달라도 시간이 비슷하게 나옴

## 2. 첫 응답 시간

스트리밍에서 첫 조각이 오기까지 걸린 시간 (`f14_1_ttfc` ~ `f14_3_ttfc`)

전체 시간은 답변 길이에 따라 달라지고, 기다린다고 느끼는 구간은 첫 글자가 나오기 전까지임. 그 구간만 따로 잼

1. `first_time = None` : 아직 받지 않은 상태로 두고 시작함
2. `if chunk.content:` : 본문이 빈 조각을 걸러냄
3. `if first_time is None:` : 첫 조각에서 한 번만 시간을 기록함
4. `break` : 첫 조각을 받은 뒤 반복을 끊음
5. `next(iter(llm.stream(prompt)))` : 반복자에서 첫 조각만 꺼냄

- `first_time is None`으로 거르지 않으면 조각마다 값을 덮어써서 마지막 조각 시간이 남음
- `break`로 끊으면 뒤 조각은 받지 않음. 첫 시간만 볼 때 씀
- `next(iter(...))`는 같은 동작을 한 줄로 적은 것
- 전체 시간과 따로 봐야 함. 첫 조각이 빨라도 끝까지는 오래 걸릴 수 있음

## 3. usage_metadata

응답에 같이 오는 토큰 사용량 (`f16_usage_metadata`, `f17_tps`)

처리량을 내려면 실제로 만든 토큰 수가 필요함. `max_tokens`로 나누면 모델이 먼저 답을 끝낸 경우 실제보다 높게 나옴

1. `response.usage_metadata` : 토큰 사용량이 든 딕셔너리
2. `response.usage_metadata["output_tokens"]` : 실제로 만든 출력 토큰 수
3. `tokens / elapsed` : 초당 토큰 수(TPS)

- `input_tokens`는 질문, `output_tokens`는 답변, `total_tokens`는 둘의 합
- **`max_tokens`를 주면 답이 끝나지 않아도 `output_tokens`가 그 값에서 멈춤**
- `input_token_details`의 `cache_read`는 다시 계산하지 않고 넘어간 입력 토큰 수
- TPS는 `max_tokens` 대신 `output_tokens`로 나눠야 실제 처리량이 됨

#### cf) 폴더로 나눈 뒤 import

- 스크립트가 늘어서 `furiosa1`·`furiosa2` 폴더로 나누면 `from common import get_llm`으로는 못 찾음
- `from furiosa1.common import get_llm`처럼 폴더 이름을 점으로 이어 적음
- `__init__.py`를 두지 않아도 폴더가 패키지로 인식됨
- 폴더마다 `common.py`를 따로 두면 그 폴더에서 쓰는 설정만 모아 둘 수 있음